<a href="https://colab.research.google.com/github/sum1t-here/pytorch-fundamentals/blob/main/002_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

y = x<sup>2</sup>

dy / dx = ?

In [4]:
# requires_grad -> whenever we want differentiation
x = torch.tensor(3.0, requires_grad=True)

x

tensor(3., requires_grad=True)

In [5]:
y = x**2

y

tensor(9., grad_fn=<PowBackward0>)

In [6]:
y.backward()

In [8]:
x.grad

tensor(6.)

y = x <sup>2</sup>

z = sin(y)

dz / dx = ?

In [10]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)
z.backward()
x.grad

tensor(-5.4668)

PyTorch builds a dynamic computation graph as each operation is executed.

Forward pass:

`x = 3 -> y = x² = 9 -> z = sin(9)`

Instead of only storing values, PyTorch also stores how each value was created.

| Tensor |	Value	grad_fn |
|-------|--------------|
| x | 	3.0	None (leaf tensor)|
| y | 	9.0	PowBackward0 |
| z | 	sin(9)	SinBackward0 |

grad_fn is the function PyTorch will use during backpropagation.

`z.backward()`

This starts reverse-mode automatic differentiation.

In [11]:
## neural network

# input
x = torch.tensor(6.7)    # input feature
y = torch.tensor(0.0)    # true label (binary)

w = torch.tensor(1.0)    # weight
b = torch.tensor(0.0)    # bias

In [12]:
## binary cross entropy loss for scalar

def binary_cross_entropy_loss(prediction, target):
  epsilon = 1e-8        # to prevent log 0 -> inf
  # clamp means "keep every value between a minimum and maximum."
  prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
  loss = - (target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))
  return loss

In [13]:
## forward pass

z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)

In [18]:
## derivative
# 1. dl/d(y_pred) : Loss wrt to prediction
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Prediction wrt z
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db
dz_dw = x
dz_db = 1

dL_dW = dloss_dy_pred * dy_pred_dz * dz_dw
dL_dB = dloss_dy_pred * dy_pred_dz * dz_db

In [19]:
print(f"Manual gradient loss for w: {dL_dW}")
print(f"Manual gradient loss for b: {dL_dB}")

Manual gradient loss for w: 6.691762447357178
Manual gradient loss for b: 0.998770534992218


In [22]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)
loss.backward()
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [ ]:
## clearing gradient -> useful in multiple pass
x.grad.zero_()

In [ ]:
## disable gradient tracking

x.requires_grad_(False)
z = x.detach()
with torch.no_grad():
  z = x + y